<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Byte-Pair-Encoder/BPE-q3.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("byte-pair-encoder", ...)`


## Problem: Write a Byte Pair Encoder in Python

### Problem Statement
Implement the **Byte Pair Encoding (BPE)** algorithm in Python. BPE is a subword tokenization method used in modern NLP models (GPT, LLaMA, etc.) that iteratively merges the most frequent pair of adjacent symbols in a vocabulary to build a compact subword vocabulary from a corpus of words.

### Requirements
1. **`get_vocab(corpus)`** — Build an initial vocabulary from a list of words:
   - Split each word into individual characters plus a special end-of-word token (e.g., `</w>`).
   - Count how many times each word appears in the corpus.

2. **`get_stats(vocab)`** — Count adjacent symbol pairs:
   - For each word in the vocabulary, count every pair of adjacent symbols.
   - Weight each pair's count by the word's frequency.

3. **`merge_vocab(pair, vocab)`** — Apply a merge operation:
   - Find every occurrence of the given pair of adjacent symbols in the vocabulary.
   - Replace the pair with a single merged symbol (concatenation of the two).
   - The merged vocabulary must spell the same words as before — only the chunking changes.

4. **`byte_pair_encoding(corpus, num_merges=10)`** — Run the full BPE loop:
   - Build the initial vocabulary, then repeatedly find and merge the most frequent pair.
   - Return the final vocabulary and the list of merge operations performed.

### Constraints
- Each merge must pick the most frequent adjacent pair (greedy).
- Merging must never change the text a word spells — only how it is segmented.
- The algorithm must be deterministic: same corpus and `num_merges` → same result.

In [ ]:
from collections import defaultdict, Counter

def get_vocab(corpus):
    """Creates a vocabulary with words split into characters and a special end-of-word token."""
    vocab ={}
    for word in corpus:
        # Split the word into characters and add a special end-of-word token
        chars = list(word) + ['</w>']
        vocab[tuple(chars)] = vocab.get(tuple(chars), 0) + 1
    return vocab

def get_stats(vocab):
    """Counts frequency of adjacent symbol pairs."""
    pairs = Counter() # Initialize the counter for symbol pairs
    for word, freq in vocab.items():
        symbols = list(word)
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merges the most frequent pair into a single symbol."""
    new_vocab = {}
    bigram = ''.join(pair)
    for word, freq in vocab.items():
        # Replace the pair with the merged symbol
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(bigram)
                i += 2  # Skip the next symbol since it's part of the merged pair
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = freq
    return new_vocab

def byte_pair_encoding(corpus, num_merges=10):
    """Performs BPE on a corpus."""
    vocab = get_vocab(corpus)
    merges = []
    for _ in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs: # pairs was an empty dictionary
            break
        most_frequent_pair = max(pairs, key=pairs.get) # most_frequent_pair = min(pairs, key=lambda p: (-pairs[p], p)) -- this is independent of the order of the pairs. this outputs the pair with the highest frequency, and in case of a tie, it will choose the pair that comes first in lexicographical order.
        vocab = merge_vocab(most_frequent_pair, vocab)
        merges.append(most_frequent_pair)
    return vocab, merges

# Example usage
corpus = ["low", "lowest", "newer", "wider"]
final_vocab, merge_operations = byte_pair_encoding(corpus, num_merges=10)

print("\nFinal Vocabulary:")
for word in final_vocab:
    print(' '.join(word), ":", final_vocab[word])



Final Vocabulary:
low</w> : 1
lowest</w> : 1
ne w er</w> : 1
w i d er</w> : 1


In [5]:
def test_get_vocab():
    corpus = ["test"]
    vocab = get_vocab(corpus)
    assert vocab == {('t', 'e', 's', 't', '</w>'): 1}
    print("✓ test_get_vocab passed")

def test_get_stats():
    vocab = {('t', 'e', 's', 't', '</w>'): 1}
    stats = get_stats(vocab)
    expected = {
        ('t', 'e'): 1,
        ('e', 's'): 1,
        ('s', 't'): 1,
        ('t', '</w>'): 1
    }
    assert stats == expected
    print("✓ test_get_stats passed")

def test_merge_vocab():
    vocab = {('t', 'e', 's', 't', '</w>'): 1}
    merged = merge_vocab(('e', 's'), vocab)
    expected = {('t', 'es', 't', '</w>'): 1}
    assert merged == expected
    print("✓ test_merge_vocab passed")

def test_bpe_sequence():
    corpus = ["low", "lower", "newest", "widest"]
    final_vocab, merges = byte_pair_encoding(corpus, num_merges=5)
    assert isinstance(final_vocab, dict)
    assert all(isinstance(pair, tuple) for pair in merges)
    assert len(merges) == 5
    print("✓ test_bpe_sequence passed")

# Run all tests
test_get_vocab()
test_get_stats()
test_merge_vocab()
test_bpe_sequence()
print("\nAll tests passed!")

✓ test_get_vocab passed
✓ test_get_stats passed
✓ test_merge_vocab passed
✓ test_bpe_sequence passed

All tests passed!
